In [ ]:
# pokemon_api_fast.py
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# FastAPI + ngrok + uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio
import os
from threading import Thread

BASE_URL = 'https://pokeapi.co/api/v2'
LIST_URL = f'{BASE_URL}/pokemon/?offset=0&limit=151'  # primeros 151

# --- Helpers para peticiones ---
def fetch_json(session: requests.Session, url: str, timeout=10):
    """Petición GET simple con manejo básico de errores."""
    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        print(f'Error al pedir {url}: {e}')
        return None

def parse_pokemon_detail(data):
    """Extrae los campos que queremos del JSON del pokemon."""
    if not data:
        return None
    name = data.get('name')
    height = data.get('height')
    weight = data.get('weight')
    base_experience = data.get('base_experience')
    # Normalizamos habilidades a lista de nombres
    abilities = [a['ability']['name'] for a in data.get('abilities', [])]
    return {
        'nombre': name,
        'altura': height,
        'peso': weight,
        'habilidades': abilities,
        'experiencia': base_experience
    }

# --- Construcción del dataset ---
def build_dataset(limit=151, max_workers=8):
    session = requests.Session()            # reusar conexiones
    # 1) obtener lista
    payload = fetch_json(session, LIST_URL)
    results = payload.get('results', []) if payload else []
    ds = []

    # 2) peticiones paralelas a cada URL de pokemon (detalle)
    urls = [p['url'] for p in results]

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_json, session, u): u for u in urls}
        for fut in as_completed(futures):
            url = futures[fut]
            data = fut.result()
            if data is None:
                # si falla una petición, podrías reintentar aquí
                print(f'Fallo al obtener detalle de {url}')
                continue
            item = parse_pokemon_detail(data)
            if item:
                ds.append(item)
            # pequeño sleep opcional para ser amable con la API pública
            time.sleep(0.02)

    # ordenar por nombre para reproducibilidad
    ds = sorted(ds, key=lambda x: x['nombre'])
    return ds

# --- Principal: crear CSV y preparar FastAPI ---
DS = build_dataset()

# Guardar CSV (habilidades como string)
df = pd.DataFrame(DS)
df['habilidades'] = df['habilidades'].apply(lambda lst: ','.join(lst))
df.to_csv('pokemon_1_151.csv', index=False)
print('CSV guardado: pokemon_1_151.csv')
print(df.head())

# --- FastAPI app ---
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/', response_class=JSONResponse)
async def root():
    # devolvemos la lista original con habilidades como lista (no como string)
    return DS

# --- Ejecutar con ngrok + uvicorn (sólo si se ejecuta directamente) ---
if __name__ == '__main__':
    nest_asyncio.apply()
    PORT = 7000
    NGROK_TOKEN = "304OCRKn9ABcVtCiwXKeP84js8H_2wBmyojCxrLDdSheyxkvN"  # opcional si ya lo configuraste
    # ================================
    # 1. Aplicar token UNA sola vez
    # ================================
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except:
        pass  # ignora si ya fue configurado antes
    # ================================
    # 2. Cerrar túneles previos (si existieran)
    # ================================
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    # Conectar ngrok (opcional)
    # ================================
    # 3. Crear un túnel nuevo
    # ================================
    ngrok_tunnel = ngrok.connect(PORT)
    public_url = ngrok_tunnel.public_url
    print('🔗 ngrok public URL:', ngrok_tunnel.public_url)
    # Ejecutar uvicorn en un hilo para no bloquear ni chocar con el loop del notebook
    def run_uvicorn():
        uvicorn.run(app, host='0.0.0.0', port=PORT)

    server_thread = Thread(target=run_uvicorn, daemon=True)
    server_thread.start()
    # ================================
    # 4. Arrancar uvicorn en hilo
    # ================================
    print("🚀 Uvicorn ejecutándose en thread...")
    print("🌍 Accede a la API en:", public_url)

Error en la solicitud: 400


In [ ]:
print("Túneles activos:", ngrok.get_tunnels())   # lista de túneles activos (si hay)

Error en la solicitud: 400
Mensaje de error: {'error': {'message': 'Invalid OAuth 2.0 Access Token', 'type': 'OAuthException', 'code': 190, 'error_subcode': 2069032, 'is_transient': False, 'error_user_title': 'No se admite el token de acceso del usuario', 'error_user_msg': 'En la nueva experiencia para páginas, se necesita un token de acceso a la página para esta llamada.', 'fbtrace_id': 'AlKGy9M-zHmpVOz0wEGUTAe'}}
